# Code to pull and clean up dataset

Plan is to pull CO2 dataset below:
https://github.com/owid/co2-data
and clean it up for analysis.

In [1]:
# imports
import os
import pandas as pd
# import matplotlib.pyplot as plt
# import numpy as np

In [2]:
# Load and cache dataset from URL if not already cached
dataset_name = "owid-co2-data"
url = "https://github.com/owid/co2-data/raw/refs/heads/master/owid-co2-data.csv"
RAW_FOLDER = "../data/raw/"
LOCAL_PATH = f"{RAW_FOLDER}/{dataset_name}.csv"

df_raw = pd.DataFrame()
if os.path.exists(LOCAL_PATH):
   df_raw = pd.read_csv(LOCAL_PATH)

else:
    df_raw = pd.read_csv(url)
    os.makedirs(os.path.dirname(RAW_FOLDER), exist_ok=True)
    df_raw.to_csv(LOCAL_PATH, index=False)

print(f"{df_raw.shape}")

(50411, 79)


In [3]:
# Also grabbing OWID Code Book with COl descriptions
url_codebook = "https://github.com/owid/co2-data/raw/refs/heads/master/owid-co2-codebook.csv"

codebook_df = pd.DataFrame()
if os.path.exists(f"{RAW_FOLDER}/owid-co2-codebook.csv"):
    codebook_df = pd.read_csv(f"{RAW_FOLDER}/owid-co2-codebook.csv")
else:
    codebook_df = pd.read_csv(url_codebook)
    codebook_df.to_csv(f"{RAW_FOLDER}/owid-co2-codebook.csv", index=False)    

In [4]:
# Initial look at the dataframe
df_raw.info()
df_raw.describe()

<class 'pandas.DataFrame'>
RangeIndex: 50411 entries, 0 to 50410
Data columns (total 79 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   country                                    50411 non-null  str    
 1   year                                       50411 non-null  int64  
 2   iso_code                                   42480 non-null  str    
 3   population                                 41167 non-null  float64
 4   gdp                                        15251 non-null  float64
 5   cement_co2                                 29173 non-null  float64
 6   cement_co2_per_capita                      25648 non-null  float64
 7   co2                                        29384 non-null  float64
 8   co2_growth_abs                             27216 non-null  float64
 9   co2_growth_prct                            26239 non-null  float64
 10  co2_including_luc                

,year,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,co2_including_luc,co2_including_luc_growth_abs,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
count,50411.000000,4.116700e+04,1.525100e+04,29173.000000,25648.000000,29384.000000,27216.000000,26239.000000,23796.000000,23496.000000,...,2170.000000,41238.000000,38280.000000,41238.000000,41238.000000,38280.000000,38150.000000,37813.000000,4712.000000,4712.000000
mean,1920.349249,6.017294e+07,3.300495e+11,7.890109,0.060026,420.227035,6.268847,42.598225,544.144592,7.483698,...,7.190616,2.272236,0.002871,0.008014,0.011224,0.000509,490.799608,310.521459,-6.986781,21.468641
std,65.859123,3.308412e+08,3.086383e+12,62.988171,0.123566,1972.092032,62.199548,1721.913018,2273.281696,99.512520,...,17.448980,9.282343,0.015362,0.045687,0.062888,0.003048,2414.076755,1812.363570,259.018184,62.637598
min,1750.000000,2.150000e+02,4.998000e+07,0.000000,0.000000,0.000000,-1928.339000,-100.000000,-84.560000,-2298.978000,...,0.000000,-0.824000,-0.001000,-0.000000,-0.001000,0.000000,-19.725000,0.000000,-2177.807000,-98.281000
25%,1875.000000,3.272140e+05,7.874038e+09,0.000000,0.000000,0.381000,-0.005000,-1.070500,6.667750,-0.727500,...,0.144000,0.003000,0.000000,0.000000,0.000000,0.000000,1.502000,0.221000,-2.262250,-6.828750
50%,1925.000000,2.291264e+06,2.743861e+10,0.000000,0.001000,5.081000,0.044000,3.813000,28.120000,0.112000,...,0.588500,0.081000,0.000000,0.000000,0.000000,0.000000,14.605500,2.222000,1.641000,8.381500
75%,1975.000000,9.986553e+06,1.212627e+11,0.524000,0.077000,53.656500,1.018000,10.884000,124.303250,2.765250,...,2.416500,0.373000,0.000000,0.001000,0.002000,0.000000,76.508500,27.863000,11.425500,32.782250
max,2024.000000,8.161973e+09,1.301126e+14,1666.885000,2.484000,38598.578000,1804.657000,180870.000000,43184.086000,2614.874000,...,100.000000,100.000000,0.377000,1.216000,1.678000,0.085000,54433.398000,43714.777000,1768.846000,1023.042000


In [5]:
# Cleaning up aggregate continent and world data from the dataset
# Rows not represented by an ISO code represent a region consisting of multiple countries
df_countries = df_raw[df_raw["iso_code"].notna()]

In [6]:
# List of South American countries
sAmerican_countries = [
    "Argentina",
    "Bolivia",
    "Brazil",
    "Chile",
    "Colombia",
    "Ecuador",
    "Guyana",
    "Paraguay",
    "Peru",
    "Suriname",
    "Uruguay",
    "Venezuela",
]
# Limit dataset to South American countries only
sAmerica_df = df_countries[df_countries["country"].isin(sAmerican_countries)]
sAmerica_df = sAmerica_df[(sAmerica_df["year"] >= 1990)  & (sAmerica_df["co2"] > 0)]

df_sur = sAmerica_df[sAmerica_df['country'] == "Suriname"]
df_sur.head()

,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
43899,Suriname,1990,SUR,412500.0,NaN,0.024,0.057,1.738,-0.114,-6.172,...,NaN,0.009,0.0,0.0,0.0,0.0,4.436,2.017,NaN,NaN
43900,Suriname,1991,SUR,415021.0,NaN,0.023,0.057,2.024,0.286,16.436,...,NaN,0.010,0.0,0.0,0.0,0.0,4.907,2.330,NaN,NaN
43901,Suriname,1992,SUR,417602.0,NaN,0.023,0.056,2.031,0.007,0.358,...,NaN,0.010,0.0,0.0,0.0,0.0,6.602,2.412,NaN,NaN
43902,Suriname,1993,SUR,419357.0,NaN,0.023,0.056,2.039,0.007,0.357,...,NaN,0.010,0.0,0.0,0.0,0.0,5.561,2.402,NaN,NaN
43903,Suriname,1994,SUR,422382.0,NaN,0.028,0.066,2.043,0.005,0.225,...,NaN,0.010,0.0,0.0,0.0,0.0,5.915,2.373,NaN,NaN


## Data Issue and proposed solution

- Guyana and Suriname GDP are completly missing in this dataset. 
- The alternative solution is to pull GDP data from World Bank data set. However, since those GDP values are expressed in international dollars at _2021_, and not _2011_ (which is what is available in the current CO2 dataset), so we will need to replace all GDP values of the CO2 dataset countries we are evaluating with the World Bank GDP values expressed in international-$ at 2021.

## Problems that would arise from this
- This will limit our analysis to only the years 1990-2025 --> will mostlikely stick to 1990-2022 for analysis since the CO2 dataset has missing values for 2023 and 2024.
- Will need to re-calculate GDP based cols in the dataset --> e,g,. co2_per_gdp

In [7]:
# Grabbing World bank dataset from OWID
world_bank_url = "https://ourworldindata.org/grapher/gdp-worldbank.csv?v=1&csvType=full&useColumnShortNames=true"

df_gdp = pd.DataFrame()
if os.path.exists(f"{RAW_FOLDER}/gdp-worldbank.csv"):
    df_gdp = pd.read_csv(f"{RAW_FOLDER}/gdp-worldbank.csv")
else:
    df_gdp = pd.read_csv(world_bank_url, storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})
    df_gdp.to_csv(f"{RAW_FOLDER}/gdp-worldbank.csv", index=False)

# checking values
df_gdp.head()

,entity,code,year,ny_gdp_mktp_pp_kd
0,Afghanistan,AFG,2000,32567375969
1,Afghanistan,AFG,2001,29495629513
2,Afghanistan,AFG,2002,37931379899
3,Afghanistan,AFG,2003,41281584746
4,Afghanistan,AFG,2004,41865355064


In [8]:
# Need to rename the columns to match the CO2 dataset for merging
df_gdp = df_gdp.rename(columns={'entity': 'country', 'code': 'iso_code', 'Year': 'year', 'ny_gdp_mktp_pp_kd': 'gdp'})
df_gdp.head()

# print(df_gdp[df_gdp['country'] == 'Suriname'])


,country,iso_code,year,gdp
0,Afghanistan,AFG,2000,32567375969
1,Afghanistan,AFG,2001,29495629513
2,Afghanistan,AFG,2002,37931379899
3,Afghanistan,AFG,2003,41281584746
4,Afghanistan,AFG,2004,41865355064


In [9]:
# Getting old dataframe ready for merge --> need to take out old GDP since it is expressed in 2011 prices
sAmerica_df = sAmerica_df.drop(columns=['gdp', 'co2_per_gdp'])

final_df = sAmerica_df.merge(df_gdp[['country', 'year', 'gdp']], how='left', on=['country', 'year'])
# Checking if there are dups
final_df.duplicated(subset=['country', 'year']).sum()


np.int64(0)

In [10]:
# Just checking if the merge worked for Suriname and Guyana
# final_df[final_df['country'] == 'Suriname'].head()

final_df[final_df['country'] == 'Guyana'].head()



,country,year,iso_code,population,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,co2_including_luc,...,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share,gdp
210,Guyana,1990,GUY,749891.0,0.0,0.0,1.129,-0.055,-4.644,3.285,...,0.040,0.0,0.0,0.0,0.0,5.609,1.714,NaN,NaN,3.805285e+09
211,Guyana,1991,GUY,747870.0,0.0,0.0,1.107,-0.022,-1.948,3.150,...,0.040,0.0,0.0,0.0,0.0,5.784,1.638,NaN,NaN,4.035805e+09
212,Guyana,1992,GUY,749919.0,0.0,0.0,1.041,-0.066,-5.960,16.420,...,0.040,0.0,0.0,0.0,0.0,14.166,1.614,NaN,NaN,4.348898e+09
213,Guyana,1993,GUY,752969.0,0.0,0.0,1.044,0.004,0.352,8.291,...,0.040,0.0,0.0,0.0,0.0,9.038,1.641,NaN,NaN,4.704425e+09
214,Guyana,1994,GUY,755821.0,0.0,0.0,1.458,0.414,39.649,13.948,...,0.039,0.0,0.0,0.0,0.0,13.113,2.025,NaN,NaN,5.105826e+09


In [11]:
# Re-calculating cols that involve GDP --> Since it is now based in 2021 buying power

# converting Million tons to Kg since co2 per gdp is expressed in Kg per international-$ in 2021 prices --> Hence multiplying by 1 milion to get the actual ton amount and then by 1000 to get Kg --> hence 1e9
final_df['co2_per_gdp'] = (final_df['co2'] * 1e9) / final_df['gdp']

# Also need to convert terrawatt hrs of primary energy consumption to kilowatt hrs for energy per gdp --> as expressed in the codebook
final_df['energy_per_gdp'] = (final_df['primary_energy_consumption'] * 1e9) / final_df['gdp']

rearrange_cols = ['country', 'year', 'population', 'co2', 'gdp','co2_per_gdp' ,'co2_per_capita','co2_growth_abs','co2_growth_prct', 'cumulative_co2', 'land_use_change_co2', 'cumulative_luc_co2',"co2_including_luc", "total_ghg", 'primary_energy_consumption', 'energy_per_gdp']

final_df = final_df[rearrange_cols]

final_df[final_df['country'] == 'Suriname'].head()



,country,year,population,co2,gdp,co2_per_gdp,co2_per_capita,co2_growth_abs,co2_growth_prct,cumulative_co2,land_use_change_co2,cumulative_luc_co2,co2_including_luc,total_ghg,primary_energy_consumption,energy_per_gdp
315,Suriname,1990,412500.0,1.738,7.069161e+09,0.245857,4.214,-0.114,-6.172,50.907,1.372,90.781,3.110,4.436,7.658,1.083297
316,Suriname,1991,415021.0,2.024,7.260028e+09,0.278787,4.877,0.286,16.436,52.931,1.403,92.184,3.427,4.907,7.454,1.026718
317,Suriname,1992,417602.0,2.031,7.289068e+09,0.278636,4.864,0.007,0.358,54.962,3.947,96.132,5.979,6.602,7.155,0.981607
318,Suriname,1993,419357.0,2.039,6.800701e+09,0.299822,4.861,0.007,0.357,57.001,2.506,98.638,4.545,5.561,7.396,1.087535
319,Suriname,1994,422382.0,2.043,7.031925e+09,0.290532,4.837,0.005,0.225,59.044,3.098,101.735,5.141,5.915,7.383,1.049926


In [12]:
# Saving clean dataset to clean data subfolder
CLEAN_FOLDER = "../data/clean"

if not os.path.exists(f"{CLEAN_FOLDER}"):
    os.makedirs(f"{CLEAN_FOLDER}", exist_ok=True)
if not os.path.exists(f"{CLEAN_FOLDER}/south_america_co2_data.csv"):
    final_df.to_csv(f"{CLEAN_FOLDER}/south_america_co2_data.csv", index=False)
